In [ ]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")


True

In [1]:
from src.preprocess.parquet_preprocessor import ParquetPreprocessor

ParquetPreprocessor.csv_to_parquet("data/raw/active_alarms_prod.csv", "data/raw/active_alarms_prod.parquet")

In [2]:
from src.preprocess.active_preprocessor import ActivePreprocessor
from src.repository.alarm_graph_repository import AlarmGraphRepository
from src.graphing.temporal_threshold_periods import TemporalThresholdPeriods

graph_repo = AlarmGraphRepository(
    uri=os.getenv("NEO4J_URI"),
    user=os.getenv("NEO4J_USER"),
    password=os.getenv("NEO4J_PASSWORD"),
    database="active"
)

lazy_frame = pl.scan_parquet("data/raw/active_alarms_prod.parquet")

lazy_frame = ActivePreprocessor.select_features(lazy_frame)
lazy_frame = ActivePreprocessor.clean_data(lazy_frame)

lazy_frame = lazy_frame.collect()


strategy = TemporalThresholdPeriods(threshold_minutes=5)

try:
    for node_df in lazy_frame.partition_by("Node ID"):
        physical_node_id = node_df["Node ID"][0]
        graph_repo.save_alarm_nodes(node_df, physical_node_id)
        graph_repo.save_temporal_edges(node_df, physical_node_id, strategy)
finally:
    graph_repo.close()


KeyboardInterrupt: 